# 💳 TP2 — Prédire l'Accord d'une Demande de Crédit (Classification)
## Module Data Science — Machine Learning | Corrigé détaillé
**Dataset** : `demandes_credit.csv` (7 000 demandes) — cible : `credit_accorde` (0/1)

In [ ]:
!pip install scikit-learn seaborn -q
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
sns.set_theme(style='whitegrid')

## 1. Exploration (EDA)

In [ ]:
df = pd.read_csv("demandes_credit.csv")
print("Dimensions :", df.shape)
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
print("Valeurs manquantes :\n", df.isna().sum())
print("\nTaux d'accord :", round(df["credit_accorde"].mean()*100, 1), "%")
print(df["credit_accorde"].value_counts())

In [ ]:
# Taux d'accord selon l'historique de crédit
plt.figure(figsize=(8,5))
df.groupby("historique_credit")["credit_accorde"].mean().sort_values().plot(kind="bar")
plt.title("Taux d'accord selon l'historique de crédit"); plt.ylabel("Taux d'accord"); plt.show()

In [ ]:
# Taux d'accord selon le type de contrat
plt.figure(figsize=(8,5))
df.groupby("type_contrat")["credit_accorde"].mean().sort_values().plot(kind="bar", color="teal")
plt.title("Taux d'accord selon le type de contrat"); plt.ylabel("Taux d'accord"); plt.show()

In [ ]:
# Revenu selon la décision
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="credit_accorde", y="revenu_mensuel")
plt.xticks([0,1], ["Refusé", "Accordé"]); plt.title("Revenu selon la décision"); plt.show()

> 🔑 Bon historique + revenu élevé → taux d'accord plus élevé. Jeu ~équilibré (~60% d'accords).

## 2. Nettoyage

In [ ]:
df["anciennete_emploi"] = df["anciennete_emploi"].fillna(df["anciennete_emploi"].median())
df["apport_personnel"]  = df["apport_personnel"].fillna(df["apport_personnel"].median())
df["historique_credit"] = df["historique_credit"].fillna(df["historique_credit"].mode()[0])
print("Manquants restants :", df.isna().sum().sum())

## 3. Préparation pour le ML

In [ ]:
df_ml = pd.get_dummies(df, columns=["type_contrat", "historique_credit", "situation_familiale"],
                       drop_first=True)
X = df_ml.drop(columns="credit_accorde")
y = df_ml["credit_accorde"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"Train : {X_train.shape[0]} | Test : {X_test.shape[0]}")

> ⚠️ On normalise pour LogReg et KNN. L'arbre n'en a pas besoin.

## 4. Les 3 modèles de classification

In [ ]:
resultats = {}
logreg = LogisticRegression(max_iter=500).fit(X_train_s, y_train)
resultats["Régression logistique"] = accuracy_score(y_test, logreg.predict(X_test_s))
arbre = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train)
resultats["Arbre de décision"] = accuracy_score(y_test, arbre.predict(X_test))
knn = KNeighborsClassifier(n_neighbors=7).fit(X_train_s, y_train)
resultats["KNN (k=7)"] = accuracy_score(y_test, knn.predict(X_test_s))
for m, s in sorted(resultats.items(), key=lambda x: x[1], reverse=True):
    print(f"{m:25} : {s:.3f}")

> 🔑 Attendu : LogReg ~0.903 (meilleur), Arbre ~0.831, KNN ~0.791.

## 5. Évaluation approfondie

In [ ]:
pred = logreg.predict(X_test_s)
cm = confusion_matrix(y_test, pred)
ConfusionMatrixDisplay(cm, display_labels=["Refusé", "Accordé"]).plot(cmap="Blues")
plt.title("Matrice de confusion — Régression logistique"); plt.show()
print(classification_report(y_test, pred, target_names=["Refusé", "Accordé"]))

## 6. Réflexion métier (Q13)

- **Faux positif** = accorder un crédit non remboursé → **perte du capital** 💸 (erreur la plus coûteuse)
- **Faux négatif** = refuser un bon client → manque à gagner

> 🔑 La banque cherche à **minimiser les faux positifs** → privilégier la **precision** de la classe « Accordé ».

## 7. Interpréter et prédire

In [ ]:
importances = pd.Series(arbre.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10,6)); importances.head(8).plot(kind="barh")
plt.title("Top 8 variables importantes"); plt.gca().invert_yaxis(); plt.show()
print(importances.head(8))

In [ ]:
def predire_credit(modele, scaler, colonnes, **infos):
    demande = pd.DataFrame(0, index=[0], columns=colonnes)
    for cle, val in infos.items():
        if cle in demande.columns:
            demande[cle] = val
    demande_s = scaler.transform(demande)
    pred  = modele.predict(demande_s)[0]
    proba = modele.predict_proba(demande_s)[0][1]
    return ("ACCORDÉ" if pred == 1 else "REFUSÉ"), proba

decision, proba = predire_credit(
    logreg, scaler, X.columns,
    age=40, revenu_mensuel=800000, anciennete_emploi=10,
    montant_demande=5000000, duree_pret_mois=36, nb_credits_actuels=0,
    apport_personnel=1500000, nb_personnes_charge=1,
    type_contrat_Fonctionnaire=1, historique_credit_Bon=1)
print(f"Profil solide → {decision} (probabilité : {proba:.1%})")

## 8. Conclusion

Meilleur modèle : **régression logistique** (~0.90). Variables clés : historique de crédit, revenu, endettement. Erreur la plus coûteuse pour la banque : le **faux positif**.